# Day 1 Lab: Economics of FinTech — Simulation Exercises

Interactive exploration of cost structures, platform dynamics, and
prediction economics

> **Expected Time**
>
> -   Exercise 1 (Structured case analysis — no code): ≈ 25 minutes
> -   Exercise 2 (Platform simulation): ≈ 40 minutes
> -   Exercise 3 (Effective fee calculator): ≈ 25 minutes
> -   Total: ≈ 90 minutes

<figure>
<a
href="https://colab.research.google.com/github/quinfer/minimba-notebooks/blob/main/lab01_economics.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

## Setup (Colab-only installs)

## Before You Start

This lab reinforces the four frameworks from today’s lectures. You do
**not** need to be a programmer to complete it. The code is provided and
runs in Google Colab; your job is to **interpret results, change
parameters, and connect findings to theory**.

The three exercises are designed in ascending order of coding
involvement:

1.  **No code** — Structured case analysis applying the economics of
    prediction
2.  **Guided simulation** — Change parameters, observe and interpret
    results
3.  **Guided computation** — Calculate and visualise fee economics

Each exercise has clear deliverables: short written interpretations
(150–300 words) that connect observations to economic theory. These
prepare you for the final assessment (briefing note).

# Exercise 1 — Economics of Prediction: Decompose a Financial Service (25 min)

## The Framework

Recall from this morning’s lecture the “anatomy of a task” (Agrawal,
Gans, and Goldfarb 2018). Every business decision can be decomposed
into:

| Component | Definition | AI impact |
|------------------------|------------------------|------------------------|
| **Prediction** | Forecasting outcomes from data and patterns | AI makes this cheap |
| **Judgement** | Determining relative payoffs, including for errors | Value *increases* as prediction cheapens |
| **Action** | Executing the chosen course | Can be automated for clear cases |
| **Data** | Input, training, and feedback information | More available, cheaper to collect |

As prediction becomes commoditised, the scarce and valuable components
are **judgement** (knowing what matters), **data curation** (knowing
what to feed the model), and **thoughtful action** (knowing when to
override).

## Your Task (No Code Required)

Choose **one** of the following financial services. For your chosen
service, complete the decomposition table below.

**Option A: Robo-advisory (e.g., Nutmeg, Wealthify, Vanguard PAS)**

**Option B: Fraud detection in card payments (e.g., Visa’s real-time
scoring)**

**Option C: Mortgage underwriting (e.g., a digital lender like Habito or
Better)**

**Option D: AML/KYC compliance screening (e.g., checking new customers
against sanctions lists)**

### Decomposition Template

Copy this table and fill it in for your chosen service:

| Component | What does this involve in your chosen service? | Who/what does it today? | Could AI improve it? | What happens if AI gets it wrong? |
|-------|------------------------|-------------|------------|------------------|
| **Prediction** |  |  |  |  |
| **Judgement** |  |  |  |  |
| **Action** |  |  |  |  |
| **Data** |  |  |  |  |

### Interpretation Questions (Write 200–300 words)

1.  **Where is the bottleneck?** Is the service expensive because
    prediction is hard, because judgement is complex, or because data is
    scarce?

2.  **Where does AI help most?** Which component benefits most from
    cheaper prediction? Which component is hardest to automate?

3.  **What is the cost of error?** If the AI prediction is wrong, what
    is the consequence? Is the consequence symmetric (equally bad in
    both directions) or asymmetric (worse in one direction)?

4.  **What automation level is appropriate?** Using the graduated
    automation scale from this morning (Level 0 = fully manual, Level 3
    = fully automated), what level would you recommend and why?

> **Hint — Direction of Answer (expand if stuck)**
>
> The bottleneck is usually **not** where AI is most publicised. For
> fraud detection, the prediction (is this transaction fraudulent?) is
> already highly automated — the bottleneck is the *judgement* threshold
> (how many false positives are acceptable, and who is accountable for
> blocked legitimate transactions). For mortgage underwriting, the
> costly step is *data acquisition* for non-standard borrowers, not
> prediction per se.
>
> On error costs: false positives (blocking legitimate transactions) and
> false negatives (allowing fraud) are *not* symmetric — the cost to
> reputation of blocking a good customer may exceed the cost of one
> fraudulent transaction. This asymmetry determines the automation
> level: where errors are asymmetric and costly, human override (Level
> 1–2) is appropriate.
>
> Full worked answers are in the separate sample answers document.

> **Connection to Day 1 Frameworks**
>
> This exercise applies the **economics of prediction** (Part II) and
> the **genuine innovation vs arbitrage** distinction (Part IV). A
> service that automates prediction where prediction is genuinely the
> bottleneck is genuine innovation. A service that automates prediction
> where judgement is the bottleneck is likely overselling AI.

# Exercise 2 — Platform Adoption Simulation (40 min)

## The Model

Two-sided platforms (card networks, payment apps, open banking) obey
coupled dynamics: growth on each side depends on the other side’s
participation, pricing, and governance quality.

The model captures four forces:

$$\text{Growth} = \underbrace{\text{Baseline}}_{\text{marketing}} + \underbrace{\text{Network effect}}_{\text{cross-side value}} - \underbrace{\text{Price friction}}_{\text{fees}} - \underbrace{\text{Congestion}}_{\text{same-side problems}}$$

Formally:

$$N_u(t+1) = N_u(t) + a_u \cdot (1 - N_u) \cdot \max(0, 1 + \beta_{um} \cdot N_m - p_u) - \gamma_u \cdot N_u^2$$

Don’t worry about the maths — focus on the four forces and what happens
when you change them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def simulate_platform(T=60, a_u=0.02, a_m=0.015,
                      beta_um=0.6, beta_mu=0.5,
                      price_u=0.0, price_m=0.03,
                      gamma_u=0.0, gamma_m=0.0):
    """
    Simulate two-sided platform adoption.
    
    Parameters:
    - T: time periods (e.g., months)
    - a_u, a_m: baseline growth rates (users, merchants)
    - beta_um, beta_mu: cross-side network effect strengths
    - price_u, price_m: fees/frictions on each side
    - gamma_u, gamma_m: same-side congestion
    
    Returns: (user_participation, merchant_participation) arrays
    """
    Nu = np.zeros(T + 1)
    Nm = np.zeros(T + 1)
    Nu[0] = 0.02
    Nm[0] = 0.02
    
    for t in range(T):
        du = a_u * (1 - Nu[t]) * max(0.0, (1 + beta_um * Nm[t] - price_u)) - gamma_u * Nu[t]**2
        dm = a_m * (1 - Nm[t]) * max(0.0, (1 + beta_mu * Nu[t] - price_m)) - gamma_m * Nm[t]**2
        Nu[t + 1] = np.clip(Nu[t] + du, 0, 1)
        Nm[t + 1] = np.clip(Nm[t] + dm, 0, 1)
    
    return Nu, Nm

print("Simulation function loaded successfully.")

## Task 2a — Four Pricing Scenarios

The economic logic: users are typically **more price-elastic** than
merchants (they would switch to cash or a rival app if charged), and
their participation creates **stronger cross-side value** for merchants
(`β_um = 0.6 > β_mu = 0.5`). These two asymmetries determine which side
to subsidise.

We set prices to values that are meaningfully large relative to the base
utility of 1.0, so the asymmetry is visible in the plots.

In [ ]:
# Price parameters are expressed as fractions of the platform's base utility (1.0).
# price_m = 0.25 means merchants pay 25% of that utility as a fee — significant but
# tolerable because cross-side effects from a large user base compensate.
# price_u = 0.40 means users pay 40% — high because users are more price-elastic
# (in real markets, even small user fees destroy adoption).
scenarios = {
    'A: Both free (no revenue)':         dict(price_u=0.00, price_m=0.00),
    'B: Merchant-pays (baseline)':       dict(price_u=0.00, price_m=0.25),
    'C: User-pays (wrong side)':         dict(price_u=0.40, price_m=0.00),
    'D: Congestion (poor governance)':   dict(price_u=0.00, price_m=0.25,
                                              gamma_u=0.15, gamma_m=0.10),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True, sharex=True)

for ax, (title, params) in zip(axes.flat, scenarios.items()):
    Nu, Nm = simulate_platform(**params)
    t = np.arange(len(Nu))
    ax.plot(t, Nu * 100, color='steelblue', linewidth=2,
            label=f'Users  → {Nu[-1]*100:.0f}%')
    ax.plot(t, Nm * 100, color='darkorange', linewidth=2, linestyle='--',
            label=f'Merchants  → {Nm[-1]*100:.0f}%')
    # Horizontal reference lines at final adoption
    ax.axhline(Nu[-1] * 100, color='steelblue',   alpha=0.25, linestyle=':')
    ax.axhline(Nm[-1] * 100, color='darkorange', alpha=0.25, linestyle=':')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(alpha=0.25, linestyle=':')
    ax.set_ylim(0, 102)

for ax in axes[1]:
    ax.set_xlabel('Time Period (months)')
for ax in axes[:, 0]:
    ax.set_ylabel('Market Participation (%)')

fig.suptitle(
    'Platform Adoption: Four Pricing Strategies\n'
    'Blue solid = users  |  Orange dashed = merchants  |  Dotted lines = final equilibrium',
    fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("Final adoption at t = 60:")
print(f"  {'Scenario':<40} {'Users':>7} {'Merchants':>11} {'Platform size':>14}")
print("  " + "-" * 74)
for title, params in scenarios.items():
    Nu, Nm = simulate_platform(**params)
    platform_size = Nu[-1] * Nm[-1] * 100
    print(f"  {title:<40} {Nu[-1]*100:>6.1f}%  {Nm[-1]*100:>10.1f}%  {platform_size:>12.1f}%²")
print("\n  'Platform size' = users × merchants (proxy for total value created)")

### Questions (Write 150–200 words)

1.  Compare panels B (merchant-pays) and C (user-pays): user adoption is
    lower in C even though merchants are free. Why? What does this
    reveal about the relative price-sensitivity of each side? (Hint:
    examine the user trajectory at t=10 vs t=30.)

2.  Panel B vs Panel A: merchant adoption is noticeably lower in B (59%)
    than A (68%), despite identical user trajectories. Why doesn’t the
    large user base fully compensate merchants for paying a 25% fee?

3.  The “both free” scenario (A) reaches the highest adoption — so why
    doesn’t every platform do this? What is the key constraint the model
    ignores?

4.  Panel D has the same pricing as B but adoption collapses to ~32%.
    What real-world problems does `gamma` represent? (Think: fraud,
    spam, low-quality listings, API abuse.) Why does poor governance
    hurt adoption even when pricing is correct?

> **Hint — Direction of Answer (expand if stuck)**
>
> Questions 1–2: The model is calibrated so users are *more*
> price-elastic than merchants (`price_u=0.40` creates more friction
> than `price_m=0.25`). In user-pays (C), early user adoption is slow
> because users face a 40% utility cost. Fewer early users → weaker
> cross-side effects for merchants → merchants also grow more slowly →
> compound disadvantage. In merchant-pays (B), users are free and adopt
> quickly, but merchants still face a 25% fee that limits their final
> ceiling even though user-side cross-side effects are strong.
>
> Question 3: The model ignores revenue altogether — “both free” works
> mathematically but requires external funding (VC subsidy) until one
> side can be monetised.
>
> Question 4: `gamma` represents same-side congestion effects — fraud
> that erodes user trust, spam that clogs developer APIs. The model
> captures: even if pricing is right, governance failure creates a
> ceiling much lower than the pricing-alone equilibrium (~32% vs ~60%).

## Task 2b — Price Sensitivity Heatmap

Now systematically explore how different combinations of user and
merchant prices affect adoption.

In [ ]:
# Scan the full range of economically meaningful prices (0 to 0.5)
price_grid = np.linspace(0.0, 0.50, 21)
Z_users     = np.zeros((len(price_grid), len(price_grid)))
Z_merchants = np.zeros((len(price_grid), len(price_grid)))
Z_platform  = np.zeros((len(price_grid), len(price_grid)))

for i, pu in enumerate(price_grid):
    for j, pm in enumerate(price_grid):
        Nu, Nm = simulate_platform(price_u=pu, price_m=pm)
        Z_users[i, j]    = Nu[-1]
        Z_merchants[i, j] = Nm[-1]
        Z_platform[i, j] = Nu[-1] * Nm[-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ['Final User Adoption', 'Final Merchant Adoption', 'Platform Size (U × M)']
arrays = [Z_users, Z_merchants, Z_platform]

for ax, arr, title in zip(axes, arrays, titles):
    im = ax.imshow(arr, origin='lower', extent=[0, 0.5, 0, 0.5],
                   aspect='auto', cmap='viridis', vmin=0, vmax=arr.max())
    plt.colorbar(im, ax=ax, label='Participation (0–1)')
    ax.set_xlabel('Merchant price (p_m)')
    ax.set_ylabel('User price (p_u)')
    ax.set_title(title, fontweight='bold')
    # Mark the four scenario points
    points = {'A\nBoth free': (0.00, 0.00),
              'B\nMerchant-pays': (0.25, 0.00),
              'C\nUser-pays': (0.00, 0.40)}
    for label, (pm_val, pu_val) in points.items():
        ax.plot(pm_val, pu_val, 'r^', markersize=8)
        ax.annotate(label, (pm_val, pu_val), textcoords='offset points',
                    xytext=(5, 5), fontsize=7, color='red')

plt.suptitle('Price Structure vs Platform Outcomes\n'
             'Red triangles = scenarios from Task 2a', fontsize=11)
plt.tight_layout()
plt.show()

### Key Observation

All three heatmaps are **asymmetric**: increasing user price (moving up)
destroys adoption and platform size far more than increasing merchant
price (moving right). The platform-size panel is the most informative —
the “sweet spot” is low user price combined with moderate merchant
price.

### Questions (Write 150–200 words)

1.  In the platform-size heatmap, where is the “sweet spot”? What
    combination of user price and merchant price maximises total
    platform value while still generating some revenue?
2.  A regulator caps merchant fees at roughly 0.20 (like the EU
    interchange cap at ~0.2%). Using the heatmap, predict what happens
    to adoption compared to unregulated (0.30–0.40). What if the
    platform responds by introducing a small user fee of 0.10?
3.  Why does the merchant-adoption heatmap look *different* from the
    user-adoption heatmap? What does this reveal about the relative
    sensitivity of each side to pricing?

> **Hint — Direction of Answer (expand if stuck)**
>
> The platform-size heatmap tells the clearest story: the “sweet spot”
> is a horizontal band near `p_u ≈ 0` and `p_m ≈ 0.20–0.30` — zero (or
> near-zero) user price combined with moderate merchant fees. Moving up
> the y-axis (raising user price) cuts platform size far more sharply
> than moving right (raising merchant price).
>
> Merchant adoption is *less* sensitive to merchant price than you might
> expect because cross-side effects from a large user base partly
> compensate. The divergence between the two heatmaps is precisely the
> *asymmetry* that makes merchant-pays the dominant design in two-sided
> markets.

## Task 2c — Network Effect Strength

Network effects determine both **speed** (how quickly you reach critical
mass) and **ceiling** (final adoption). We use the same pricing as Task
2a scenario B (merchant-pays, `price_m=0.25`) to isolate the effect of
network strength.

In [ ]:
# Panel 1: Trajectories at four beta values — shows how S-curve shape changes
beta_showcase = [0.2, 0.4, 0.6, 0.8]
colors_beta = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for b, col in zip(beta_showcase, colors_beta):
    Nu, Nm = simulate_platform(beta_um=b, beta_mu=b * 0.8, price_m=0.25)
    t = np.arange(len(Nu))
    axes[0].plot(t, Nu * 100, color=col, linewidth=2, label=f'β={b:.1f}  (final {Nu[-1]*100:.0f}%)')
    axes[0].plot(t, Nm * 100, color=col, linewidth=1.5, linestyle='--')

axes[0].axhline(50, color='grey', linestyle=':', linewidth=1.5, label='50% critical mass')
axes[0].set_title('S-Curve Shape at Different β Values\n(solid=users, dashed=merchants)', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Time Period')
axes[0].set_ylabel('Market Participation (%)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 100)
axes[0].grid(alpha=0.25, linestyle=':')

# Panel 2: Time to reach 50% user adoption vs beta
betas_full = np.linspace(0.1, 0.9, 33)
t50_users = []
t50_merchants = []
finals = []

for b in betas_full:
    Nu, Nm = simulate_platform(beta_um=b, beta_mu=b * 0.8, price_m=0.25, T=120)
    t50_users.append(next((t for t, n in enumerate(Nu) if n >= 0.50), 120))
    t50_merchants.append(next((t for t, n in enumerate(Nm) if n >= 0.50), 120))
    finals.append(Nu[-1] * 100)

axes[1].plot(betas_full, t50_users, 'o-', linewidth=2, color='steelblue', label='Users reach 50%')
axes[1].plot(betas_full, t50_merchants, 's--', linewidth=2, color='darkorange', label='Merchants reach 50%')
axes[1].set_title('Periods to Reach 50% Adoption vs Network Effect Strength\n(shorter = faster scale-up)', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Cross-side network effect strength (β)')
axes[1].set_ylabel('Periods to reach 50% adoption')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.25, linestyle=':')
axes[1].invert_yaxis()  # lower = faster — inverted so "better" is up

fig.suptitle('Network Effects: Speed and Ceiling', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Speed-to-50% summary:")
for b in [0.2, 0.4, 0.6, 0.8]:
    Nu, Nm = simulate_platform(beta_um=b, beta_mu=b*0.8, price_m=0.25, T=120)
    t50u = next((t for t, n in enumerate(Nu) if n >= 0.50), '>120')
    print(f"  β={b:.1f}: users reach 50% at period {t50u:>4},  final adoption {Nu[-1]*100:.0f}%")

### Questions (Write 100–150 words)

1.  In the left panel, compare the S-curves at β=0.2 and β=0.8. What is
    the key difference — is it the final adoption level, the *shape* of
    the curve, or the *speed*? What does this mean for a startup trying
    to challenge an established platform with strong network effects?

2.  The right panel inverts the y-axis so “better” is higher (fewer
    periods to reach critical mass). At what approximate β value does
    the speed of scale-up become meaningfully faster? What does this
    imply for platform strategy — should you focus on growing network
    effects or cutting price?

3.  For Open Banking, β_um represents how much app developers value
    broad bank API coverage. The UK’s CMA mandated the 9 largest banks
    to provide APIs. What did this policy do to β in the model’s terms?

> **Hint — Direction of Answer (expand if stuck)**
>
> The key difference at β=0.2 vs β=0.8 is primarily **speed** (time to
> critical mass), not the eventual ceiling — both converge to high
> adoption eventually. This is the crucial insight: a competitor doesn’t
> need a fundamentally different product to dislodge an incumbent; they
> need to achieve critical mass *before* the incumbent’s network effects
> compound further.
>
> On the regulatory question: mandating bank API coverage effectively
> raised β_um in the model — it increased the cross-side value of being
> a developer on the platform. This is the “cold-start solution by
> regulation” argument. But the heatmap from Task 2b shows that raising
> β alone does not guarantee adoption; trust, UX, and compelling use
> cases (the qualitative factors the model can’t capture) matter too.

# Exercise 3 — The Economics of Automation: Fee Structures (25 min)

## Why This Matters

The economics of prediction framework says: when prediction becomes
cheap, the value shifts to judgement. Robo-advisors are the clearest
financial example — they automate prediction (portfolio optimisation,
rebalancing) while leaving judgement (risk tolerance, life goals) to the
client or a human advisor.

But the *cost structure* reveals something additional: traditional
advisors impose minimum fees that make small accounts prohibitively
expensive. Automation doesn’t just make advice cheaper — it **expands
access** to people who were previously excluded.

## Task 3a — Effective Fee Rates

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

account_sizes = np.array([10_000, 25_000, 50_000, 100_000, 250_000, 500_000])
labels = [f'£{s//1000:.0f}k' for s in account_sizes]

# Traditional adviser: 1.5% AUM, £2,500 annual minimum
trad_rate = 0.015
trad_min  = 2_500
trad_fee  = np.maximum(trad_rate * account_sizes, trad_min)

# Robo-adviser: 0.25% AUM, no minimum
robo_rate = 0.0025
robo_fee  = robo_rate * account_sizes

# Derived metrics
trad_effective    = trad_fee / account_sizes * 100   # effective % rate
robo_effective    = robo_fee / account_sizes * 100
effective_savings = trad_effective - robo_effective   # regressive-structure story
min_binding_size  = trad_min / trad_rate              # £166,667

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel 1: Effective fee rates + minimum binding annotation ────────────────
axes[0].plot(account_sizes / 1000, trad_effective, 'o-', linewidth=2,
             color='tab:blue', label='Traditional (1.5% + £2,500 min)')
axes[0].plot(account_sizes / 1000, robo_effective, 's-', linewidth=2,
             color='tab:red', label='Robo (0.25%, no min)')
axes[0].axvline(min_binding_size / 1000, color='grey', linestyle='--', linewidth=1.2)
axes[0].annotate(f'Minimum stops\nbinding at\n£{min_binding_size/1000:.0f}k',
                 xy=(min_binding_size/1000, 5),
                 xytext=(min_binding_size/1000 + 10, 12),
                 arrowprops=dict(arrowstyle='->', color='grey'),
                 fontsize=8, color='grey')
axes[0].fill_between(account_sizes / 1000, robo_effective, trad_effective,
                     alpha=0.12, color='tab:blue', label='Cost of staying traditional')
axes[0].set_xlabel('Account Size (£000s)')
axes[0].set_ylabel('Effective Annual Fee Rate (%)')
axes[0].set_title('Flat Minimums Create a Regressive Fee Structure\n(shaded = cost of not switching to robo)', fontsize=10)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3, linestyle=':')

# ── Panel 2: Effective savings RATE (%) — who gains MOST from switching? ─────
bars = axes[1].bar(labels, effective_savings, color='tab:green', edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, effective_savings):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                 f'{val:.2f}%', ha='center', fontsize=9, fontweight='bold')
axes[1].axvline(3.5, color='grey', linestyle='--', linewidth=1.2)
axes[1].text(3.7, effective_savings.max() * 0.85, '← minimum\nbinding',
             fontsize=8, color='grey')
axes[1].set_ylabel('Effective Savings Rate (% of assets per year)')
axes[1].set_title('Small Accounts Gain the Most from Switching\n(savings as % of assets, not absolute £)', fontsize=10)
axes[1].grid(axis='y', alpha=0.3, linestyle=':')

plt.suptitle('Robo vs Traditional Advisor: Fee Economics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
print(f"{'Account':>10} {'Trad fee':>10} {'Robo fee':>10} {'Trad eff%':>10} {'Robo eff%':>10} {'Saving%':>9}")
print("  " + "-" * 62)
for i, a in enumerate(account_sizes):
    print(f"  £{a:>8,} £{trad_fee[i]:>8,.0f} £{robo_fee[i]:>8,.0f}"
          f"  {trad_effective[i]:>8.2f}%  {robo_effective[i]:>8.2f}%  {effective_savings[i]:>8.2f}%")
print(f"\n  Minimum ceases to bind above: £{min_binding_size:,.0f}")
print(f"  Trad is never cheaper than robo on rate alone (1.5% > 0.25% always)")

### Questions (Write 150–200 words)

1.  The right panel shows **effective savings rate** (% of assets per
    year), not absolute pounds. Why is this the right metric for
    comparing fee impact? A £10k account saves 24.75% of its value per
    year by switching — what does that mean for a typical investor?

2.  The left panel shows a shaded region that *narrows* as account size
    grows. What structural feature of traditional advisory pricing
    creates this wedge for small accounts? Why don’t traditional
    advisors simply drop their minimum?

3.  The effective rate for traditional advisory stabilises at exactly
    1.5% above £166,667. Connect this to the economics of prediction:
    robo-advisors automate *prediction* (portfolio allocation).
    Traditional advisors provide *judgement* (tax planning, behavioural
    coaching, estate planning). At what wealth level does that 1.25%
    premium for human judgement become worthwhile?

> **Hint — Direction of Answer (expand if stuck)**
>
> Question 1: Absolute savings are dominated by large accounts (£6,250
> for £500k vs £2,475 for £10k) and hide the *access* problem. Effective
> rate savings show that small accounts are most disadvantaged: 24.75%
> vs 1.25% effective drag. A typical small investor paying a 25%
> effective fee on their portfolio is not getting 25% of value in
> services — they’re paying a fixed infrastructure cost (the advisor’s
> overhead per client), regardless of account size.
>
> Question 2: Traditional advisors can’t drop their minimum because
> their cost structure is human-intensive — a client meeting, annual
> review, compliance documentation, and liability insurance cost roughly
> the same whether the account holds £10k or £500k. Robo-advisors break
> this model because software marginal cost is near-zero.
>
> Question 3: The judgement premium (1.25% p.a.) is worth it if the
> advisor adds value exceeding that cost: complex tax planning (ISA/SIPP
> optimisation worth thousands per year at high wealth), estate
> planning, behavioural coaching (preventing panic-selling). These
> become relevant at roughly £200k–£300k where the absolute fee is
> significant but the complexity of the financial situation justifies
> it.

## Task 3b — Compounding the Savings (Optional Extension)

If you invest the fee savings at 7% annual return, how much extra wealth
do you accumulate over 20 years? We compare **three account sizes** to
reveal which investors benefit most.

In [ ]:
years = 20
annual_return = 0.07

accounts = {
    '£10k (excluded by min)': 10_000,
    '£50k (mass market)':     50_000,
    '£250k (affluent)':       250_000,
}
colors_3 = ['tab:red', 'tab:blue', 'tab:green']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trajectories = {}
for (label, account), col in zip(accounts.items(), colors_3):
    trad_annual  = max(trad_rate * account, trad_min)
    robo_annual  = robo_rate * account
    annual_saving = trad_annual - robo_annual

    cumulative = 0
    traj = []
    for y in range(years):
        cumulative = (cumulative + annual_saving) * (1 + annual_return)
        traj.append(cumulative)
    trajectories[label] = (traj, annual_saving, account)

    axes[0].plot(range(1, years + 1), traj, 'o-', linewidth=2, color=col,
                 label=f'{label}  (£{annual_saving:,.0f}/yr saving)')

axes[0].set_xlabel('Year')
axes[0].set_ylabel('Cumulative Extra Wealth (£)')
axes[0].set_title('Compounding Fee Savings Over 20 Years\n(invested at 7% p.a.)', fontsize=10, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3, linestyle=':')

# Panel 2: Extra wealth as % of original account — who benefits proportionally most?
final_pcts = []
lab_list = []
for (label, account), col in zip(accounts.items(), colors_3):
    traj, _, _ = trajectories[label]
    pct = traj[-1] / account * 100
    final_pcts.append(pct)
    lab_list.append(label)
    axes[1].bar(label, pct, color=col)
    axes[1].text(lab_list.index(label), pct + 1,
                 f'{pct:.0f}%\nof account', ha='center', fontsize=9, fontweight='bold')

axes[1].set_ylabel('Extra wealth after 20 yrs (% of original account)')
axes[1].set_title('Extra Wealth as % of Starting Account\n(shows proportional access benefit)', fontsize=10, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3, linestyle=':')

plt.suptitle('Who Benefits Most from Switching to Robo?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Summary:")
print(f"  {'Account':<30} {'Annual saving':>14} {'Extra wealth (20yr)':>20} {'As % of account':>16}")
print("  " + "-" * 82)
for (label, account), col in zip(accounts.items(), colors_3):
    traj, saving, _ = trajectories[label]
    pct = traj[-1] / account * 100
    print(f"  {label:<30} £{saving:>12,.0f}   £{traj[-1]:>17,.0f}   {pct:>14.0f}%")
print("\n  Note: £10k account actually accumulates MORE than £50k in absolute terms")
print("  because the flat minimum creates a higher absolute fee drag at small account sizes.")

This reveals a **counterintuitive result**: the £10k account (which
traditional advisors effectively exclude with a £2,500 minimum)
accumulates *more* extra wealth in absolute terms than the £50k account,
because the minimum creates a larger annual saving. Automation removes
an access barrier that disproportionately penalises small investors.

# Synthesis — Connecting the Exercises

## The Through-Line

These three exercises illustrate a single argument:

1.  **Exercise 1** (prediction decomposition): Identified *where* in a
    financial service AI can reduce costs and *where* human judgement
    remains essential.

2.  **Exercise 2** (platform simulation): Showed *how* platform dynamics
    determine which business models succeed — pricing structure matters
    more than pricing level, and governance matters as much as pricing.

3.  **Exercise 3** (fee economics): Demonstrated *who benefits* when
    automation reduces costs — and that the largest gains accrue to
    previously excluded participants (small accounts).

Together, these exercises operationalise Day 1’s frameworks: the cost
puzzle (Exercise 3 shows where costs compress), the economics of
prediction (Exercise 1 decomposes the value chain), platform economics
(Exercise 2 shows scaling dynamics), and the evidence lens (all
exercises require you to interpret results critically, not just accept
outputs).

## Final Reflection (200–300 words)

Write a short reflection connecting your findings across all three
exercises:

1.  Which financial services are most susceptible to FinTech disruption?
    Why? (Use your Exercise 1 decomposition and Exercise 3 cost
    analysis.)
2.  What role do platform dynamics play in determining which FinTech
    innovations succeed? (Use your Exercise 2 simulation findings.)
3.  Where is the line between genuine innovation and regulatory
    arbitrage? How would you tell the difference?

This reflection is direct preparation for the final assessment (briefing
note).

## References

Agrawal, Ajay, Joshua Gans, and Avi Goldfarb. 2018. *Prediction
Machines: The Simple Economics of Artificial Intelligence*. Harvard
Business Review Press.